# Phase 1: Country/supplier concentration risk for rare earth magnets

Same core idea as the pharma-shortage-risk project: concentration of supply
is the risk signal. Here, that means how few countries supply most of the
world's rare earth elements, and specifically how few countries the US
actually imports magnet-grade heavy rare earths from right now.

This notebook builds the concentration metrics, sanity-checks them against
the real 2025 export-control shock (documented production impacts at BMW,
Ford, and Suzuki), and flags an important data-quality nuance before
drawing conclusions from it.

In [1]:
import sys
sys.path.insert(0, '../src')
import concentration_risk as cr
import pandas as pd
pd.set_option('display.max_colwidth', 100)

## 1. Mine production and reserve concentration, by country

In [2]:
prod_res = cr.load_production_reserves()
conc = pd.concat([
    cr.concentration_by_year(prod_res, 'mine_production_tons_reo'),
    cr.concentration_by_year(prod_res, 'reserves_tons_reo'),
])
conc

,source_report,data_year,metric,n_countries,top1_country,top1_share_pct,hhi
0,MCS2025,2023,mine_production_tons_reo,11,China,32.24,2029.6
2,MCS2026,2024,mine_production_tons_reo,12,China,70.50,5220.1
1,MCS2025,2024e,mine_production_tons_reo,11,China,34.18,2086.9
3,MCS2026,2025e,mine_production_tons_reo,12,China,69.45,5086.5
0,MCS2025,2023,reserves_tons_reo,8,Australia,60.54,4234.6
2,MCS2026,2024,reserves_tons_reo,11,China,51.59,3370.8
1,MCS2025,2024e,reserves_tons_reo,8,Australia,60.54,4234.6
3,MCS2026,2025e,reserves_tons_reo,11,China,51.59,3370.8


### Important caveat before reading this table as a trend

China is the top producer in every snapshot, that part is solid and matches
every other source reviewed for this project. But notice the production HHI
roughly doubles between the MCS2025 edition (top1_share ~32-34%) and the
MCS2026 edition (top1_share ~69-70%) for similarly-labeled years.

That jump is **not** a real-world shift that happened during 2025. Looking
at the raw country data, Burma's reported production falls from 143,000 to
27,000 tons and Madagascar's from 112,100 to 1,400 tons between the two
report editions, almost certainly a methodology revision in how USGS
estimates informal/artisanal mining output in those countries, not an
actual collapse in their mining activity. Reserve figures show a similar
edition-to-edition jump for Australia (125.7M tons in MCS2025 down to
6.3M tons in MCS2026), again a reserve-classification revision, not a
discovery of reserves disappearing.

**Takeaway: compare concentration numbers within a single report edition,
never across editions, and don't read the edition-to-edition jump as a 2025
supply-chain event.** The real 2025 event is documented separately below,
from trade/export data, not from these annual production tables.

## 2. Heavy rare earth import-source concentration (the magnet-relevant one)

Mine production concentration is the broad picture. For EV motor magnets
specifically, what matters more is where the US actually imports the
finished heavy rare earth elements from, since magnet manufacturing adds
dysprosium and terbium to neodymium magnets to keep them magnetic at the
high operating temperatures inside a motor.

In [3]:
heavy = cr.heavy_ree_risk_tiers(cr.load_heavy_ree_import_reliance())
heavy

,element,magnet_relevant,china_share_pct,risk_tier,notes
0,Terbium,yes_hre_magnet_additive,100.0,severe (single-source dependent),Used with dysprosium in neodymium magnets to retain magnetism at high motor operating temperatures
1,Dysprosium,yes_hre_magnet_additive,NaN,unknown (not broken out in source table),Not broken out in this table's import-source list; treated separately below via secondary sourcing
2,Holmium,no,100.0,severe (single-source dependent),NaN
3,Lutetium,no,100.0,severe (single-source dependent),NaN
4,Erbium,no,51.0,high (majority single-source),China is secondary source at 40%
5,Ytterbium,no,86.0,high (majority single-source),"Also Republic of Korea 3%, other 3%"


Terbium, holmium, and lutetium are all at 100% US import reliance on
China, "severe" by this project's tiering. Terbium is the one of these
that's actually used in magnets. Dysprosium, the other magnet-critical
heavy rare earth, isn't broken out in this particular USGS import-source
table, so no number is reported for it here rather than guessing one.

## 3. US net import reliance over time: a genuinely declining trend

This is the one honest piece of good news in the data, and it would be
dishonest to leave it out just because it complicates a "crisis" story.

In [4]:
cr.load_us_import_reliance_trend()

,source_report,year,net_import_reliance_pct
5,MCS2026,2021,>95
6,MCS2026,2022,>95
7,MCS2026,2023,>90
8,MCS2026,2024,53
9,MCS2026,2025e,67


US net import reliance on rare earths fell from over 95% (2021-2022) to
roughly 53-67% (2024-2025e), likely reflecting domestic separation/processing
capacity (such as MP Materials' Mountain Pass facility) ramping up. That's a
real, partial offset against the concentration risk shown above, not
something to paper over.

## 4. Sanity check: does this concentration picture match a real event?

A concentration score is only useful if it would have flagged real trouble.
In 2025, China tightened rare earth magnet export controls. Here's what
actually happened, from secondary reporting (Wood Mackenzie, labeled as
such since it isn't primary government data):

In [5]:
cr.load_shock_events()

,metric,value,period,source
0,china_magnet_export_decline_pct,51,April 2025 vs March 2025,Wood Mackenzie (via supply chain press reporting)
1,china_export_license_approval_rate_pct,25,2025 (automotive suppliers),Wood Mackenzie (via supply chain press reporting)
2,us_magnet_import_decline_pct,58,2025,Wood Mackenzie (via supply chain press reporting)
3,india_magnet_import_decline_pct,78,2025,Wood Mackenzie (via supply chain press reporting)
4,south_korea_magnet_import_decline_pct,73,2025,Wood Mackenzie (via supply chain press reporting)
5,germany_magnet_import_decline_pct,47,2025,Wood Mackenzie (via supply chain press reporting)
6,china_global_processing_share_pct,90,2025,"Wood Mackenzie analyst statement (processing capacity, distinct from mine production share)"
7,us_domestic_alt_supply_investment_usd,1400000000,2025,Wood Mackenzie (via supply chain press reporting)
8,automakers_with_reported_production_impact,BMW; Ford; Suzuki,2025,Wood Mackenzie / European Association of Automotive Suppliers (via supply chain press reporting)


This lines up with the concentration picture: the elements with the most
concentrated supply (terbium, holmium, lutetium at 100% China reliance) sit
inside the same magnet supply chain that saw a 51% one-month export drop, a
25% export-license approval rate, and documented production impacts at BMW,
Ford, and Suzuki. The concentration metric would have flagged this risk
before it happened, which is the actual point of building it.

## 5. Where this connects to BMW specifically

BMW's Spartanburg, South Carolina plant is mid-ramp on EV production and a
battery assembly plant in nearby Woodruff, SC is completing in 2026, both of
which depend on exactly the permanent-magnet motor technology that uses
these import-concentrated heavy rare earths. That connection, and what a
buffer-stock response might look like, is built out in Phase 2.